[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Fisica_de_Plasmas_y_Fusion_%28PLA%29/Plasmas_%28physics.plasm-ph%29/PLA-03_Simulacion_Tokamak_MHD_Confinamiento.ipynb)

# [PLA-03] Plasmas: Magnetohidrodinámica en Tokamak

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Física de Plasmas: Confinamiento Magnético MHD en un Reactor Tokamak
# Simulamos la trayectoria de un protón atrapado magnéticamente en un toroide.
# Fuerza de Lorentz: F = q*(v x B)

# Parámetros del plasma y del reactor
q_m = 1.0 # Relación carga/masa 
R0 = 5.0  # Radio mayor del Tokamak (Toroide)
B0 = 2.0  # Intensidad magnética toroidal
dt = 0.01
steps = 2000

# Campo magnético analítico idealizado de un Tokamak
# B_toroidal decae como 1/R. B_poloidal se añade para retorcer las líneas.
def B_field(x, y, z):
    R = np.sqrt(x**2 + y**2)
    # Campo Toroidal (dirección azimutal)
    Bt = B0 * R0 / R
    phi = np.arctan2(y, x)
    Bx = -Bt * np.sin(phi)
    By =  Bt * np.cos(phi)
    
    # Campo Poloidal (Para evitar derivas, crea el helicoide)
    # Aproximación de corriente de plasma uniforme
    r_minor = np.sqrt((R - R0)**2 + z**2)
    Bp = 0.5 * r_minor # Magnitud poloidal
    if r_minor == 0: return np.array([Bx, By, 0])
    
    theta = np.arctan2(z, R - R0)
    # Vector poloidal
    Bx += -Bp * np.sin(theta) * np.cos(phi)
    By += -Bp * np.sin(theta) * np.sin(phi)
    Bz =   Bp * np.cos(theta)
    
    return np.array([Bx, By, Bz])

# Condiciones iniciales del protón
r_pos = np.array([R0 + 1.0, 0.0, 0.0]) # Desplazado del centro
v_vel = np.array([0.0, 5.0, 1.0])      # Velocidad inicial (térmica)

trajectory = np.zeros((steps, 3))

# Integrador simple de Runge-Kutta / Newton-Lorentz
for i in range(steps):
    trajectory[i] = r_pos
    B_vec = B_field(*r_pos)
    
    # Aceleración por Lorentz: a = (q/m)(v x B)
    acc = q_m * np.cross(v_vel, B_vec)
    
    v_vel += acc * dt
    r_pos += v_vel * dt

trajectory = trajectory.T

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(trajectory[0], trajectory[1], trajectory[2], lw=1.5, color='orange', label="Órbita Ciclotrónica (Protón)")

# Dibujar la pared de la "dona" del Tokamak para contexto
u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, 2 * np.pi, 20)
U, V = np.meshgrid(u, v)
X_toro = (R0 + 2*np.cos(V)) * np.cos(U)
Y_toro = (R0 + 2*np.cos(V)) * np.sin(U)
Z_toro = 2 * np.sin(V)
ax.plot_wireframe(X_toro, Y_toro, Z_toro, color='blue', alpha=0.1)

ax.set_title("Magnetohidrodinámica: Confinamiento de Plasma (Tokamak)", fontsize=14)
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.legend()
plt.show()

# FÍSICA: Observa cómo el protón realiza pequeños círculos rápidos (Giroradio de Larmor)
# mientras la "Cuerda" magnética central lo fuerza a girar eternamente alrededor 
# del interior del toroide sin tocar nunca las paredes (Confinamiento Ideal).

